# Lumpy 05 — Future Forecasting: Full SKU History vs Collision-Flag Only

This notebook produces an apples-to-apples comparison of two lumpy collision-demand scopes:

1. **`lumpy_all_sku_history`** — uses the established Lumpy 02 result and does not repeat model selection.
2. **`lumpy_collision_flag_only`** — runs a fresh model comparison on the shorter history.

Confirmed operating setup:

- full-history winner loaded from `results/lumpy_outputs/tables`;
- flag-only selection metric: mean **3-month rolling WMAPE** (the same metric used by Lumpy 03);
- backtest test window: **18 months**;
- operational gap: **3 months**;
- future horizon: **18 months**, beginning after the 3-month gap;
- five charts: the best five common SKUs by average backtest WMAPE across the two winning forecasts.

Future accuracy is not reported as WMAPE because future actuals do not yet exist. Backtest WMAPE is shown beside the future forecasts as the evidence supporting each selected model.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Locate the Lumpy Fellas repo whether the notebook starts from the repo root
# or its notebooks directory.
start = Path.cwd().resolve()
project_root = None
for candidate in [start, start.parent, start.parent.parent]:
    if (candidate / "src" / "lumpy_forecasting.py").exists():
        project_root = candidate
        break
if project_root is None:
    raise FileNotFoundError("Could not locate the Lumpy Fellas repo and src/lumpy_forecasting.py")

src_path = str(project_root / "src")
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
# Force a fresh import from the detected repo path. Notebook kernels often
# keep an older lumpy_forecasting module cached from another notebook.
sys.modules.pop("lumpy_forecasting", None)
lf = importlib.import_module("lumpy_forecasting")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

SKU_COLUMN = lf.SKU_COLUMN
DATE_COLUMN = lf.MONTH_COLUMN
TARGET_COLUMN = lf.TARGET_COLUMN

FORECAST_LAG_MONTHS = 3
BACKTEST_MONTHS = 18
FUTURE_MONTHS = 18

OUTPUT_DIRECTORY = project_root / "results" / "lumpy_05_future_forecasting"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
REVIEW_PACK_DIRECTORY = project_root / "docs" / "lumpy_05_review_pack"
REVIEW_PACK_DIRECTORY.mkdir(parents=True, exist_ok=True)
LUMPY_TABLE_DIRECTORY = project_root / "results" / "lumpy_outputs" / "tables"

# Use the release's normal candidates plus its explicit optional diagnostics.
FLAG_ONLY_CANDIDATE_KEYS = tuple(dict.fromkeys(
    lf.DEFAULT_MODEL_NAMES + lf.OPTIONAL_MODEL_NAMES
))
MODEL_KEY_BY_LABEL = {
    "Zero Forecast": "zero",
    "Recent Mean 6m": "recent_mean_6",
    "SBA Croston": "sba_croston",
    "SBA Croston Tuned": "sba_croston_tuned",
    "Seasonal SBA Croston": "seasonal_sba_croston",
    "TSB": "tsb",
    "Boosted SBA Hybrid": "boosted_sba_hybrid",
    "Aggregate Allocation": "aggregate_allocation",
    "Hurdle Random Forest": "hurdle_random_forest",
}
available_model_keys = set(lf.DEFAULT_MODEL_NAMES + lf.OPTIONAL_MODEL_NAMES)
missing_model_keys = sorted(set(MODEL_KEY_BY_LABEL.values()) - available_model_keys)
if missing_model_keys:
    raise ValueError(
        "Lumpy 05 loaded an out-of-date lumpy_forecasting module from "
        f"{Path(lf.__file__).resolve()}. Missing model keys: {missing_model_keys}"
    )

print("Using lumpy module:", Path(lf.__file__).resolve())
print("Raw outputs:", OUTPUT_DIRECTORY.resolve())
print("Review pack:", REVIEW_PACK_DIRECTORY.resolve())

## 1. Load both repo variants and the established full-history result

The two duplicate filenames are resolved through their variant folders. The full-history winner is read from Lumpy 02's saved 18-month / 3-month-lag summary rather than retrained here.

In [ ]:
full_config = lf.LumpyConfig(
    variant="all_sku_history",
    test_months=BACKTEST_MONTHS,
    gap_months=FORECAST_LAG_MONTHS,
)
flag_config = lf.LumpyConfig(
    variant="collision_flag_only",
    test_months=BACKTEST_MONTHS,
    gap_months=FORECAST_LAG_MONTHS,
    min_train_months=6,
)

full_sales, external_raw = lf.load_lumpy_inputs(project_root, full_config)
flag_sales, _ = lf.load_lumpy_inputs(project_root, flag_config)
raw_scopes = {
    "lumpy_all_sku_history": full_sales,
    "lumpy_collision_flag_only": flag_sales,
}

monthly_summary_path = LUMPY_TABLE_DIRECTORY / "lumpy_monthly_total_model_summary.csv"
saved_forecasts_path = LUMPY_TABLE_DIRECTORY / "lumpy_backtest_forecasts.csv"
if not monthly_summary_path.exists() or not saved_forecasts_path.exists():
    raise FileNotFoundError(
        "Run lumpy_02_forecasting.ipynb before Lumpy 05; its saved summary and forecasts are required."
    )

saved_monthly_summary = pd.read_csv(monthly_summary_path)
full_required_summary = saved_monthly_summary.loc[
    saved_monthly_summary["test_months"].eq(BACKTEST_MONTHS)
    & saved_monthly_summary["gap_months"].eq(FORECAST_LAG_MONTHS)
].sort_values(["mean_rolling_3m_wmape_percent", "model"])
if full_required_summary.empty:
    raise ValueError("Lumpy 02 has no saved 18-month / 3-month-lag result.")

full_winner_row = full_required_summary.iloc[0]
FULL_HISTORY_WINNER = full_winner_row["model"]
FULL_HISTORY_WINNER_KEY = MODEL_KEY_BY_LABEL[FULL_HISTORY_WINNER]
FULL_HISTORY_ROLLING_WMAPE = float(full_winner_row["mean_rolling_3m_wmape_percent"])

scope_summary = pd.DataFrame([
    {
        "scope": scope,
        "rows": len(frame),
        "skus": frame[SKU_COLUMN].nunique(),
        "first_month": frame[DATE_COLUMN].min(),
        "last_month": frame[DATE_COLUMN].max(),
        "months": frame[DATE_COLUMN].nunique(),
        "total_demand": frame[TARGET_COLUMN].sum(),
        "zero_row_share_percent": 100 * frame[TARGET_COLUMN].eq(0).mean(),
    }
    for scope, frame in raw_scopes.items()
])

scope_summary.to_csv(OUTPUT_DIRECTORY / "lumpy_05_scope_summary.csv", index=False)
display(scope_summary)
display(full_required_summary)
print(f"Reusing full-history winner: {FULL_HISTORY_WINNER} ({FULL_HISTORY_ROLLING_WMAPE:.3f}%)")

## 2. Complete each SKU-month grid and create features

Each scope receives its own grid and features. This prevents pre-flag demand from leaking into the collision-flag-only model.

In [ ]:
def prepare_scope(raw_scope):
    model_data, external_inventory = lf.build_lumpy_model_frame(
        raw_scope,
        external_raw,
        full_config,
    )
    return model_data, external_inventory


prepared_scopes = {}
for scope, raw_scope in raw_scopes.items():
    scope_config = full_config if scope == "lumpy_all_sku_history" else flag_config
    model_data, external_inventory = lf.build_lumpy_model_frame(
        raw_scope, external_raw, scope_config
    )
    prepared_scopes[scope] = {
        "completed": model_data,
        "model_data": model_data,
        "external_inventory": external_inventory,
        "config": scope_config,
    }
    print(
        scope,
        "| months:", model_data[DATE_COLUMN].nunique(),
        "| SKUs:", model_data[SKU_COLUMN].nunique(),
        "| rows:", len(model_data),
    )

## 3. Reuse the full-history winner; select only the flag-only winner

Full-history evidence comes directly from Lumpy 02. Only the collision-flag-only scope runs a new 18-month test with a 3-month operational gap. Models are ranked on `mean_rolling_3m_wmape_percent`, matching Lumpy 03.

The short collision-flag-only run is expected to have roughly seven training months. Its winner is retained for the requested comparison, but its metric is marked **directional / low confidence**.

In [ ]:
def make_required_split(data, scope):
    config = flag_config if scope == "lumpy_collision_flag_only" else full_config
    splits = lf.make_backtest_splits(data, config).copy()
    if splits.empty:
        raise ValueError(f"No valid 18-month / 3-month-lag split for {scope}.")
    splits["global_fold_id"] = np.arange(1, len(splits) + 1)
    return splits


def rank_models_on_rolling_wmape(backtest_forecasts):
    monthly = lf.summarize_monthly_totals(backtest_forecasts)
    _, ranking = lf.summarize_monthly_total_tables(monthly)
    ranking = ranking.sort_values(["mean_rolling_3m_wmape_percent", "model"]).reset_index(drop=True)
    ranking["rolling_3m_average_wmape_percent"] = ranking["mean_rolling_3m_wmape_percent"]
    ranking["rank"] = np.arange(1, len(ranking) + 1)
    return monthly, ranking


saved_full_forecasts = pd.read_csv(
    saved_forecasts_path,
    usecols=[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN, "forecast", "model", "test_months", "gap_months"],
    parse_dates=[DATE_COLUMN],
)
saved_full_forecasts = saved_full_forecasts.loc[
    saved_full_forecasts["test_months"].eq(BACKTEST_MONTHS)
    & saved_full_forecasts["gap_months"].eq(FORECAST_LAG_MONTHS)
    & saved_full_forecasts["model"].eq(FULL_HISTORY_WINNER)
].copy()

saved_monthly_detail = pd.read_csv(
    LUMPY_TABLE_DIRECTORY / "lumpy_monthly_total_results.csv",
    parse_dates=[DATE_COLUMN],
)
full_monthly = saved_monthly_detail.loc[
    saved_monthly_detail["test_months"].eq(BACKTEST_MONTHS)
    & saved_monthly_detail["gap_months"].eq(FORECAST_LAG_MONTHS)
    & saved_monthly_detail["model"].eq(FULL_HISTORY_WINNER)
].copy()
full_ranking = full_winner_row.to_frame().T.copy()
full_ranking.insert(0, "scope", "lumpy_all_sku_history")
full_ranking["rolling_3m_average_wmape_percent"] = FULL_HISTORY_ROLLING_WMAPE
full_ranking["rank"] = 1
full_ranking["confidence"] = "established_lumpy_02_result"
full_ranking["train_months"] = np.nan

flag_scope = "lumpy_collision_flag_only"
flag_bundle = prepared_scopes[flag_scope]
flag_splits = make_required_split(flag_bundle["model_data"], flag_scope)
flag_forecasts = lf.run_lumpy_backtest(
    flag_bundle["model_data"],
    flag_splits,
    flag_config,
    model_names=FLAG_ONLY_CANDIDATE_KEYS,
)
flag_monthly, flag_ranking = rank_models_on_rolling_wmape(flag_forecasts)
flag_ranking.insert(0, "scope", flag_scope)
flag_ranking["confidence"] = "directional_low_confidence"
flag_ranking["train_months"] = int(flag_splits["train_months"].min())

backtest_results = {
    "lumpy_all_sku_history": {
        "splits": pd.DataFrame(),
        "forecasts": saved_full_forecasts,
        "monthly": full_monthly,
        "ranking": full_ranking,
        "winner": FULL_HISTORY_WINNER,
        "winner_key": FULL_HISTORY_WINNER_KEY,
    },
    flag_scope: {
        "splits": flag_splits,
        "forecasts": flag_forecasts,
        "monthly": flag_monthly,
        "ranking": flag_ranking,
        "winner": flag_ranking.iloc[0]["model"],
        "winner_key": MODEL_KEY_BY_LABEL[flag_ranking.iloc[0]["model"]],
    },
}

model_comparison = pd.concat(
    [result["ranking"] for result in backtest_results.values()],
    ignore_index=True,
)
winner_comparison = model_comparison.loc[model_comparison["rank"].eq(1)].copy()

model_comparison.to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_model_comparison.csv", index=False
)
model_comparison.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_model_comparison_all_models.csv", index=False
)
winner_comparison.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_best_models_by_scope.csv", index=False
)
pd.concat([
    result["monthly"].assign(scope=scope)
    for scope, result in backtest_results.items()
], ignore_index=True).to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_backtest_monthly.csv", index=False
)

display(winner_comparison)
display(model_comparison)

## 4. Refit each winner and forecast 18 months after the 3-month gap

If the latest actual is April 2026, the gap is May–July 2026 and the first forecast month is August 2026. No zero-calibration layer is applied.

In [ ]:
def build_future_raw(completed_history, future_dates):
    latest_by_sku = (
        completed_history.sort_values([SKU_COLUMN, DATE_COLUMN])
        .groupby(SKU_COLUMN, as_index=False)
        .tail(1)
        .drop(columns=[DATE_COLUMN, TARGET_COLUMN], errors="ignore")
    )
    dates = pd.DataFrame({DATE_COLUMN: future_dates})
    latest_by_sku["_join_key"] = 1
    dates["_join_key"] = 1
    future = latest_by_sku.merge(dates, on="_join_key", how="inner").drop(columns="_join_key")
    future[TARGET_COLUMN] = 0.0  # placeholder only; future WMAPE is never calculated
    return future.sort_values([SKU_COLUMN, DATE_COLUMN]).reset_index(drop=True)


def fit_winner_and_forecast(scope, bundle, result):
    train = bundle["model_data"].copy()
    last_actual_month = pd.Timestamp(train[DATE_COLUMN].max())
    forecast_start = last_actual_month + pd.DateOffset(months=FORECAST_LAG_MONTHS + 1)
    future_dates = pd.date_range(forecast_start, periods=FUTURE_MONTHS, freq="MS")

    future_raw = build_future_raw(raw_scopes[scope], future_dates)
    future_features, _ = lf.build_lumpy_model_frame(
        future_raw,
        external_raw,
        bundle["config"],
    )

    winner = result["winner"]
    forecast = lf.run_model(
        result["winner_key"],
        train,
        future_features,
        bundle["config"],
    )
    forecast["scope"] = scope
    forecast["selected_model"] = winner
    forecast["last_actual_month"] = last_actual_month
    forecast["gap_months"] = FORECAST_LAG_MONTHS
    forecast["forecast_start"] = forecast_start
    forecast["forecast_horizon_months"] = FUTURE_MONTHS
    forecast["backtest_rolling_3m_average_wmape_percent"] = float(
        result["ranking"].iloc[0]["rolling_3m_average_wmape_percent"]
    )
    forecast["backtest_confidence"] = result["ranking"].iloc[0]["confidence"]
    return forecast


future_results = {}
future_frames = []
for scope, bundle in prepared_scopes.items():
    forecast = fit_winner_and_forecast(
        scope,
        bundle,
        backtest_results[scope],
    )
    future_results[scope] = {
        "forecast": forecast,
    }
    future_frames.append(forecast)

future_forecasts = pd.concat(future_frames, ignore_index=True, sort=False)
future_forecasts.to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_future_forecasts_by_sku.csv", index=False
)

future_monthly = (
    future_forecasts.groupby(["scope", DATE_COLUMN], as_index=False)
    .agg(
        forecast_total=("forecast", "sum"),
        forecast_positive_skus=("forecast", lambda values: int((values > 0).sum())),
        selected_model=("selected_model", "first"),
        backtest_rolling_3m_average_wmape_percent=(
            "backtest_rolling_3m_average_wmape_percent", "first"
        ),
        backtest_confidence=("backtest_confidence", "first"),
    )
)

future_side_by_side = future_monthly.pivot(
    index=DATE_COLUMN,
    columns="scope",
    values="forecast_total",
).reset_index()
future_side_by_side.columns.name = None
future_side_by_side["absolute_forecast_difference"] = np.abs(
    future_side_by_side["lumpy_all_sku_history"]
    - future_side_by_side["lumpy_collision_flag_only"]
)
future_side_by_side.to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_future_monthly_side_by_side.csv", index=False
)
future_monthly.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_future_monthly_by_scope.csv", index=False
)
future_side_by_side.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_future_monthly_side_by_side.csv", index=False
)

display(winner_comparison)
display(future_monthly)
display(future_side_by_side)

## 5. Choose the five best common SKUs by backtest WMAPE

The ranking uses only SKUs with positive backtest actual demand in both scopes. “Best” means the lowest mean WMAPE across the two respective winning models.

In [ ]:
def winning_sku_backtest_metrics(scope, result):
    winner = result["winner"]
    winning_rows = result["forecasts"].loc[
        result["forecasts"]["model"].eq(winner)
    ].copy()
    winning_rows["absolute_error"] = np.abs(
        winning_rows[TARGET_COLUMN] - winning_rows["forecast"]
    )
    metrics = (
        winning_rows.groupby(SKU_COLUMN, as_index=False)
        .agg(
            actual_total=(TARGET_COLUMN, "sum"),
            forecast_total=("forecast", "sum"),
            absolute_error=("absolute_error", "sum"),
            actual_positive_months=(TARGET_COLUMN, lambda values: int((values > 0).sum())),
            forecast_positive_months_gt_0=("forecast", lambda values: int((values > 0).sum())),
            forecast_positive_months_gt_1=("forecast", lambda values: int((values > 1).sum())),
            backtest_rows=(DATE_COLUMN, "size"),
        )
    )
    metrics["forecast_bias"] = metrics["forecast_total"] - metrics["actual_total"]
    metrics["sku_wmape_percent"] = np.where(
        metrics["actual_total"].gt(0),
        100 * metrics["absolute_error"] / metrics["actual_total"],
        np.nan,
    )
    metrics["forecast_bias_percent"] = np.where(
        metrics["actual_total"].gt(0),
        100 * metrics["forecast_bias"] / metrics["actual_total"],
        np.nan,
    )
    metrics["wmape_valid"] = metrics["actual_total"].gt(0)
    metrics["zero_actual_forecast_total"] = np.where(
        metrics["actual_total"].gt(0), 0.0, metrics["forecast_total"]
    )
    metrics["wmape_band"] = pd.cut(
        metrics["sku_wmape_percent"],
        bins=[-np.inf, 50, 70, 100, 200, np.inf],
        labels=["under_50", "50_to_70", "70_to_100", "100_to_200", "over_200"],
    ).astype("object")
    metrics.loc[~metrics["wmape_valid"], "wmape_band"] = "no_actual_demand"
    return metrics.assign(scope=scope, selected_model=winner)


sku_backtest_metrics = pd.concat([
    winning_sku_backtest_metrics(scope, result)
    for scope, result in backtest_results.items()
], ignore_index=True)


def write_scope_sku_review_file(scope, output_name):
    scope_metrics = sku_backtest_metrics.loc[sku_backtest_metrics["scope"].eq(scope)].copy()
    future_summary = (
        future_forecasts.loc[future_forecasts["scope"].eq(scope)]
        .groupby(SKU_COLUMN, as_index=False)
        .agg(
            future_18m_forecast_total=("forecast", "sum"),
            future_18m_average_monthly_forecast=("forecast", "mean"),
            future_18m_max_monthly_forecast=("forecast", "max"),
            future_18m_forecast_months=(DATE_COLUMN, "nunique"),
            future_18m_forecast_months_gt_0=("forecast", lambda values: int((values > 0).sum())),
            future_18m_forecast_months_gt_1=("forecast", lambda values: int((values > 1).sum())),
            future_selected_model=("selected_model", "first"),
        )
    )
    review = scope_metrics.merge(future_summary, on=SKU_COLUMN, how="left")
    review = review.sort_values([
        "wmape_valid",
        "sku_wmape_percent",
        "actual_total",
        SKU_COLUMN,
    ], ascending=[False, True, False, True]).reset_index(drop=True)
    ordered_columns = [
        SKU_COLUMN,
        "scope",
        "selected_model",
        "actual_total",
        "forecast_total",
        "absolute_error",
        "forecast_bias",
        "sku_wmape_percent",
        "forecast_bias_percent",
        "wmape_valid",
        "wmape_band",
        "backtest_rows",
        "actual_positive_months",
        "forecast_positive_months_gt_0",
        "forecast_positive_months_gt_1",
        "zero_actual_forecast_total",
        "future_18m_forecast_total",
        "future_18m_average_monthly_forecast",
        "future_18m_max_monthly_forecast",
        "future_18m_forecast_months",
        "future_18m_forecast_months_gt_0",
        "future_18m_forecast_months_gt_1",
        "future_selected_model",
    ]
    review = review[[column for column in ordered_columns if column in review.columns]]
    review.to_csv(REVIEW_PACK_DIRECTORY / output_name, index=False)
    return review

full_history_all_skus_wmape = write_scope_sku_review_file(
    "lumpy_all_sku_history",
    "lumpy_05_full_history_all_skus_wmape.csv",
)
short_history_all_skus_wmape = write_scope_sku_review_file(
    "lumpy_collision_flag_only",
    "lumpy_05_short_history_all_skus_wmape.csv",
)

sku_review_file_summary = pd.DataFrame([
    {
        "file": "lumpy_05_full_history_all_skus_wmape.csv",
        "scope": "lumpy_all_sku_history",
        "sku_count": len(full_history_all_skus_wmape),
        "valid_wmape_skus": int(full_history_all_skus_wmape["wmape_valid"].sum()),
        "zero_actual_skus": int((~full_history_all_skus_wmape["wmape_valid"]).sum()),
    },
    {
        "file": "lumpy_05_short_history_all_skus_wmape.csv",
        "scope": "lumpy_collision_flag_only",
        "sku_count": len(short_history_all_skus_wmape),
        "valid_wmape_skus": int(short_history_all_skus_wmape["wmape_valid"].sum()),
        "zero_actual_skus": int((~short_history_all_skus_wmape["wmape_valid"]).sum()),
    },
])
sku_review_file_summary.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_sku_wmape_file_summary.csv", index=False
)

sku_wmape_wide = sku_backtest_metrics.pivot(
    index=SKU_COLUMN,
    columns="scope",
    values="sku_wmape_percent",
).dropna().reset_index()
sku_wmape_wide.columns.name = None
sku_wmape_wide["mean_scope_wmape_percent"] = sku_wmape_wide[[
    "lumpy_all_sku_history",
    "lumpy_collision_flag_only",
]].mean(axis=1)



sku_metric_wide = sku_backtest_metrics.pivot_table(
    index=SKU_COLUMN,
    columns="scope",
    values=["actual_total", "forecast_total", "absolute_error", "sku_wmape_percent", "selected_model"],
    aggfunc="first",
)
sku_metric_wide.columns = [f"{metric}_{scope}" for metric, scope in sku_metric_wide.columns]
sku_metric_wide = sku_metric_wide.reset_index()
required_wmape_columns = [
    "sku_wmape_percent_lumpy_all_sku_history",
    "sku_wmape_percent_lumpy_collision_flag_only",
]
top_50_common_skus = sku_metric_wide.dropna(subset=required_wmape_columns).copy()
top_50_common_skus["mean_scope_wmape_percent"] = top_50_common_skus[required_wmape_columns].mean(axis=1)
top_50_common_skus = (
    top_50_common_skus.sort_values([
        "mean_scope_wmape_percent",
        "sku_wmape_percent_lumpy_all_sku_history",
        "sku_wmape_percent_lumpy_collision_flag_only",
        SKU_COLUMN,
    ])
    .head(50)
    .reset_index(drop=True)
)

top_50_future_summary = (
    future_forecasts.loc[future_forecasts[SKU_COLUMN].isin(top_50_common_skus[SKU_COLUMN])]
    .groupby([SKU_COLUMN, "scope"], as_index=False)
    .agg(
        future_18m_forecast_total=("forecast", "sum"),
        future_18m_months=(DATE_COLUMN, "nunique"),
        future_selected_model=("selected_model", "first"),
    )
    .pivot(index=SKU_COLUMN, columns="scope")
)
top_50_future_summary.columns = [f"{metric}_{scope}" for metric, scope in top_50_future_summary.columns]
top_50_future_summary = top_50_future_summary.reset_index()
top_50_common_skus = top_50_common_skus.merge(top_50_future_summary, on=SKU_COLUMN, how="left")

def _winning_backtest_rows_for_query(scope, result):
    winner = result["winner"]
    rows = result["forecasts"].loc[result["forecasts"]["model"].eq(winner)].copy()
    rows = rows.loc[rows[SKU_COLUMN].isin(top_50_common_skus[SKU_COLUMN])]
    rows = rows[[SKU_COLUMN, DATE_COLUMN, TARGET_COLUMN, "forecast", "model"]].copy()
    rows["scope"] = scope
    rows["selected_model"] = winner
    rows["period"] = "backtest_window"
    rows = rows.rename(columns={TARGET_COLUMN: "actual"})
    return rows

backtest_query_rows = pd.concat([
    _winning_backtest_rows_for_query(scope, result)
    for scope, result in backtest_results.items()
], ignore_index=True)
future_query_rows = future_forecasts.loc[
    future_forecasts[SKU_COLUMN].isin(top_50_common_skus[SKU_COLUMN]),
    [SKU_COLUMN, DATE_COLUMN, "forecast", "scope", "selected_model"],
].copy()
future_query_rows["actual"] = np.nan
future_query_rows["model"] = future_query_rows["selected_model"]
future_query_rows["period"] = "future_18m_forecast"
sku_query_rows = pd.concat([
    backtest_query_rows,
    future_query_rows[[SKU_COLUMN, DATE_COLUMN, "actual", "forecast", "model", "scope", "selected_model", "period"]],
], ignore_index=True, sort=False).sort_values([SKU_COLUMN, "scope", "period", DATE_COLUMN])

top_50_common_skus.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_top_50_common_skus_by_wmape.csv", index=False
)
sku_query_rows.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_top_50_sku_backtest_and_future_monthly.csv", index=False
)


top_five_skus = (
    sku_wmape_wide.sort_values([
        "mean_scope_wmape_percent",
        "lumpy_all_sku_history",
        "lumpy_collision_flag_only",
        SKU_COLUMN,
    ])
    .head(5)
    .reset_index(drop=True)
)

sku_backtest_metrics.to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_winner_backtest_sku_metrics.csv", index=False
)
top_five_skus.to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_top_five_common_skus.csv", index=False
)
display(top_five_skus)

## 6. Five side-by-side SKU forecast graphs

Each graph shows both observed histories and both future forecasts. Dashed vertical lines mark the end of actuals and the beginning of the forecast after the operational gap.

In [ ]:
SCOPE_COLOURS = {
    "lumpy_all_sku_history": "#1f77b4",
    "lumpy_collision_flag_only": "#d62728",
}


def plot_sku_comparison(sku, rank_row):
    fig, ax = plt.subplots(figsize=(14, 5.5))

    for scope, bundle in prepared_scopes.items():
        history = bundle["completed"].loc[
            bundle["completed"][SKU_COLUMN].eq(sku),
            [DATE_COLUMN, TARGET_COLUMN],
        ].sort_values(DATE_COLUMN)
        forecast = future_results[scope]["forecast"].loc[
            future_results[scope]["forecast"][SKU_COLUMN].eq(sku),
            [DATE_COLUMN, "forecast"],
        ].sort_values(DATE_COLUMN)

        ax.plot(
            history[DATE_COLUMN],
            history[TARGET_COLUMN],
            color=SCOPE_COLOURS[scope],
            alpha=0.55,
            linewidth=1.5,
            label=f"Actual — {scope}",
        )
        ax.plot(
            forecast[DATE_COLUMN],
            forecast["forecast"],
            color=SCOPE_COLOURS[scope],
            linewidth=2.5,
            marker="o",
            markersize=3,
            label=f"Future forecast — {scope}",
        )

    last_actual = max(
        bundle["model_data"][DATE_COLUMN].max()
        for bundle in prepared_scopes.values()
    )
    first_forecast = min(
        result["forecast"][DATE_COLUMN].min()
        for result in future_results.values()
    )
    ax.axvline(last_actual, color="black", linestyle="--", alpha=0.7, label="Last actual")
    ax.axvline(first_forecast, color="grey", linestyle=":", alpha=0.9, label="Forecast start")
    ax.axvspan(last_actual, first_forecast, color="grey", alpha=0.08, label="3-month gap")

    full_wmape = rank_row["lumpy_all_sku_history"]
    flag_wmape = rank_row["lumpy_collision_flag_only"]
    ax.set_title(
        f"SKU {sku} | backtest WMAPE: full history {full_wmape:.1f}% | "
        f"flag only {flag_wmape:.1f}%"
    )
    ax.set_xlabel("Month")
    ax.set_ylabel("Demand / forecast")
    ax.grid(alpha=0.2)
    ax.legend(loc="upper left", ncol=2, fontsize=8)
    fig.tight_layout()

    safe_sku = str(sku).replace("/", "_").replace("\\", "_")
    figure_path = OUTPUT_DIRECTORY / f"lumpy_05_top_sku_{safe_sku}.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()
    return figure_path


chart_paths = [
    plot_sku_comparison(row[SKU_COLUMN], row)
    for _, row in top_five_skus.iterrows()
]
display(pd.DataFrame({"chart_path": chart_paths}))

## 7. Final comparison summary and audit checks

The short-history result is intentionally retained but never presented as equally reliable evidence. These checks also ensure the future dates begin after exactly three skipped months and contain exactly 18 forecast months per scope.

In [ ]:
audit_rows = []
for scope, result in future_results.items():
    forecast = result["forecast"]
    last_actual = pd.Timestamp(forecast["last_actual_month"].iloc[0])
    expected_start = last_actual + pd.DateOffset(months=FORECAST_LAG_MONTHS + 1)
    actual_start = pd.Timestamp(forecast[DATE_COLUMN].min())
    audit_rows.append({
        "scope": scope,
        "selected_model": forecast["selected_model"].iloc[0],
        "last_actual_month": last_actual,
        "forecast_start": actual_start,
        "expected_forecast_start": expected_start,
        "start_date_check": actual_start == expected_start,
        "forecast_months": forecast[DATE_COLUMN].nunique(),
        "horizon_check": forecast[DATE_COLUMN].nunique() == FUTURE_MONTHS,
        "sku_count": forecast[SKU_COLUMN].nunique(),
        "forecast_total": forecast["forecast"].sum(),
        "backtest_rolling_3m_average_wmape_percent": (
            forecast["backtest_rolling_3m_average_wmape_percent"].iloc[0]
        ),
        "backtest_confidence": forecast["backtest_confidence"].iloc[0],
    })

final_comparison_summary = pd.DataFrame(audit_rows)
final_comparison_summary.to_csv(
    OUTPUT_DIRECTORY / "lumpy_05_final_comparison_summary.csv", index=False
)
final_comparison_summary.to_csv(
    REVIEW_PACK_DIRECTORY / "lumpy_05_final_comparison_summary.csv", index=False
)

if not final_comparison_summary[["start_date_check", "horizon_check"]].all().all():
    raise AssertionError("A future forecast date or horizon audit check failed.")

display(final_comparison_summary)
print("All lumpy 05 raw outputs written to:", OUTPUT_DIRECTORY.resolve())
print("Review pack written to:", REVIEW_PACK_DIRECTORY.resolve())